In [54]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
#from dotenv import load_dotenv; load_dotenv('C:\git-repository\Blog\.gitignore\.env')
from dotenv import load_dotenv; load_dotenv("C:\git-repository\BLS_prediction_project\.gitignore\.env")
import requests
import fredapi as fa
import seaborn as sns
import statsmodels.api as sm

<>:6: SyntaxWarning: invalid escape sequence '\g'
<>:6: SyntaxWarning: invalid escape sequence '\g'
C:\Users\paulp\AppData\Local\Temp\ipykernel_12392\1123047401.py:6: SyntaxWarning: invalid escape sequence '\g'
  from dotenv import load_dotenv; load_dotenv("C:\git-repository\BLS_prediction_project\.gitignore\.env")


#### DATA WRANGLING & CLEANING

In [87]:
# NON-FARM PAYROLLS -- MONTHLY

fred_key = os.getenv('fred')
base_url = 'https://api.stlouisfed.org/fred/'
obs_endpoint = 'series/observations'

series_id = 'PAYEMS'
start_date = '2000-01-01'
end_date = '2026-05-01'
ts_frequency = 'm'
ts_units = 'lin'

payems_params = {
    'series_id': series_id,
    'api_key': fred_key,
    'file_type': 'json',
    'observation_start': start_date,
    'observation_end': end_date,
    'frequency': ts_frequency,
    'units': ts_units
}

response = requests.get(base_url + obs_endpoint, params=payems_params)
data = response.json()
payems_df = pd.DataFrame(data['observations'])
payems_df['date'] = pd.to_datetime(payems_df['date'])
payems_df['value'] = pd.to_numeric(payems_df['value'], errors='coerce')
payems_df = (
    payems_df
    .set_index('date')[['value']]
    .resample('ME')
    .last()     # USE LAST OBSERVATION WITH PAYROLLS, AVERAGES DILUTE THE SIGNAL
    .rename(columns={'value': 'payems'})
)

payems_df.head()

,payems
date,
2000-01-31,131011
2000-02-29,131120
2000-03-31,131604
2000-04-30,131884
2000-05-31,132105


In [72]:
# UNEMPLOYMENT CLAIMS -- WEEKLY

series_id = 'ICSA'
start_date = '2000-01-01'
end_date = '2026-05-01'
ts_frequency = 'w'
ts_units = 'lin'

unemp_claims_params = {
    'series_id': series_id,
    'api_key': fred_key,
    'file_type': 'json',
    'observation_start': start_date,
    'observation_end': end_date,
    'frequency': ts_frequency,
    'units': ts_units
}

response = requests.get(base_url + obs_endpoint, params=unemp_claims_params)
data = response.json()
unemp_claims_df = pd.DataFrame(data['observations'])

unemp_claims_df['date'] = pd.to_datetime(unemp_claims_df['date'])
unemp_claims_df['value'] = pd.to_numeric(unemp_claims_df['value'], errors='coerce')

unemp_claims_df = (
    unemp_claims_df
    .set_index('date')[['value']]
    .resample('ME')
    .mean()
    .rename(columns={'value': 'unemp_claims'})
)

unemp_claims_df.head()

# unemp_claims_df = unemp_claims_df.set_index('date')
# unemp_claims_df = unemp_claims_df.resample('M').last()
# print(unemp_claims_df.dtypes)



,unemp_claims
date,
2000-01-31,288400.0
2000-02-29,293750.0
2000-03-31,274750.0
2000-04-30,271600.0
2000-05-31,282250.0


In [ ]:
# CONTINUING CLAIMS -- WEEKLY

series_id = 'CCSA'
start_date = '2000-01-01'
end_date = '2026-05-01'
ts_frequency = 'w'
ts_units = 'lin'

cont_claims_params = {
    'series_id': series_id,
    'api_key': fred_key,
    'file_type': 'json',
    'observation_start': start_date,
    'observation_end': end_date,
    'frequency': ts_frequency,
    'units': ts_units
}

response = requests.get(base_url + obs_endpoint, params=cont_claims_params)
data = response.json()
cont_claims_df = pd.DataFrame(data['observations'])

cont_claims_df['date'] = pd.to_datetime(cont_claims_df['date'])
cont_claims_df['value'] = pd.to_numeric(cont_claims_df['value'], errors='coerce')

cont_claims_df = (
    cont_claims_df
    .set_index('date')[['value']]
    .resample('ME')
    .mean()
    .rename(columns={'value': 'cont_claims'})
)

cont_claims_df.head()

,cont_claims
date,
2000-01-31,2109800.0
2000-02-29,2147500.0
2000-03-31,2079000.0
2000-04-30,2015400.0
2000-05-31,1989500.0


In [ ]:
# MARKET YIELD ON 2-YEAR TREASURY SECURITIES -- DAILY

series_id = 'DGS2'
start_date = '2000-01-01'
end_date = '2026-05-01'
ts_frequency = 'd'
ts_units = 'lin'

two_year_yield_params = {
    'series_id': series_id,
    'api_key': fred_key,
    'file_type': 'json',
    'observation_start': start_date,
    'observation_end': end_date,
    'frequency': ts_frequency,
    'units': ts_units
}

response = requests.get(base_url + obs_endpoint, params=two_year_yield_params)
data = response.json()
two_year_yield_df = pd.DataFrame(data['observations'])
two_year_yield_df['date'] = pd.to_datetime(two_year_yield_df['date'])
two_year_yield_df['value'] = pd.to_numeric(two_year_yield_df['value'], errors='coerce')

two_year_yield_df = (
    two_year_yield_df
    .set_index('date')[['value']]
    .resample('ME')
    .last()     # USE LAST OBSERVATION WITH YIELDS, AVERAGES DILUTE THE SIGNAL
    .rename(columns={'value': 'two_year_yield'})
)
two_year_yield_df.head()

,two_year_yield
date,
2000-01-31,6.61
2000-02-29,6.53
2000-03-31,6.50
2000-04-30,6.68
2000-05-31,6.69


In [ ]:
# MARKET YIELD ON 10-YEAR TREASURY SECURITIES -- DAILY

series_id = 'DGS10'
start_date = '2000-01-01'
end_date = '2026-05-01'
ts_frequency = 'd'
ts_units = 'lin'

ten_year_yield_params = {
    'series_id': series_id,
    'api_key': fred_key,
    'file_type': 'json',
    'observation_start': start_date,
    'observation_end': end_date,
    'frequency': ts_frequency,
    'units': ts_units
}

response = requests.get(base_url + obs_endpoint, params=ten_year_yield_params)
data = response.json()  
ten_year_yield_df = pd.DataFrame(data['observations'])

ten_year_yield_df['date'] = pd.to_datetime(ten_year_yield_df['date'])
ten_year_yield_df['value'] = pd.to_numeric(ten_year_yield_df['value'], errors='coerce')

ten_year_yield_df = (
    ten_year_yield_df
    .set_index('date')[['value']]
    .resample('ME')
    .last()
    .rename(columns={'value': 'ten_year_yield'})
)

ten_year_yield_df.head()

,ten_year_yield
date,
2000-01-31,6.68
2000-02-29,6.42
2000-03-31,6.03
2000-04-30,6.23
2000-05-31,6.29


In [ ]:
# 10 YEAR TREASURY YIELD SPREAD -- DAILY

series_id = 'T10Y2Y'
start_date = '2000-01-01'
end_date = '2026-05-01'
ts_frequency = 'd'
ts_units = 'lin'

ten_year_treasury_params = {
    'series_id': series_id,
    'api_key': fred_key,
    'file_type': 'json',
    'observation_start': start_date,
    'observation_end': end_date,
    'frequency': ts_frequency,
    'units': ts_units
}

response = requests.get(base_url + obs_endpoint, params=ten_year_treasury_params)
data = response.json()  
ten_year_treasury_df = pd.DataFrame(data['observations'])

ten_year_treasury_df['date'] = pd.to_datetime(ten_year_treasury_df['date'])
ten_year_treasury_df['value'] = pd.to_numeric(ten_year_treasury_df['value'], errors='coerce')

ten_year_treasury_df = (
    ten_year_treasury_df
    .set_index('date')[['value']]
    .resample('ME')
    .last()
    .rename(columns={'value': 'ten_year_treasury_spread'})
)
ten_year_treasury_df.head()

,ten_year_treasury_spread
date,
2000-01-31,0.07
2000-02-29,-0.11
2000-03-31,-0.47
2000-04-30,-0.45
2000-05-31,-0.40


In [ ]:
# UNIVERSITY OF MICHIGAN: CONSUMER SENTIMENT -- MONTHLY

series_id = 'UMCSENT'
start_date = '2000-01-01'
end_date = '2026-05-01'
ts_frequency = 'm'
ts_units = 'lin'

sentiment_params = {
    'series_id': series_id,
    'api_key': fred_key,
    'file_type': 'json',
    'observation_start': start_date,
    'observation_end': end_date,
    'frequency': ts_frequency,
    'units': ts_units
}

response = requests.get(base_url + obs_endpoint, params=sentiment_params)
data = response.json()
sentiment_df = pd.DataFrame(data['observations'])
sentiment_df['date'] = pd.to_datetime(sentiment_df['date'])
sentiment_df['value'] = pd.to_numeric(sentiment_df['value'], errors='coerce')
sentiment_df = (
    sentiment_df
    .set_index('date')[['value']]
    .resample('ME')
    .last()
    .rename(columns={'value': 'sentiment'})
)
sentiment_df['sentiment_lag1'] = sentiment_df['sentiment'].shift(1) # LAGGED SENTIMENT TO AVOID LOOKAHEAD BIAS IN MODELING
sentiment_df.head()

,sentiment,sentiment_lag1
date,,
2000-01-31,112.0,NaN
2000-02-29,111.3,112.0
2000-03-31,107.1,111.3
2000-04-30,109.2,107.1
2000-05-31,110.7,109.2


In [89]:
fred_data_df = (
    payems_df
    .join(unemp_claims_df, how='outer')
    .join(cont_claims_df, how='outer')  
    .join(two_year_yield_df, how='outer')
    .join(ten_year_yield_df, how='outer')
    .join(ten_year_treasury_df, how='outer')
    .join(sentiment_df, how='outer')

)

fred_data_df.describe()
#fred_data_df.head()

,payems,unemp_claims,cont_claims,two_year_yield,ten_year_yield,ten_year_treasury_spread,sentiment,sentiment_lag1
count,316.000000,3.160000e+02,3.160000e+02,317.000000,317.000000,317.000000,315.000000,314.000000
mean,140653.768987,3.693807e+05,3.039510e+06,2.266435,3.316246,1.049811,81.642222,81.732484
std,9268.015649,3.222431e+05,2.159085e+06,1.759850,1.290699,0.965067,14.086899,14.017851
min,129706.000000,1.975000e+05,1.365250e+06,0.110000,0.550000,-1.060000,50.000000,50.000000
25%,132342.250000,2.385625e+05,1.938950e+06,0.640000,2.230000,0.220000,71.150000,71.500000
50%,137748.000000,3.218750e+05,2.565500e+06,1.840000,3.440000,1.010000,82.700000,82.800000
75%,148315.000000,3.968625e+05,3.440562e+06,3.840000,4.300000,1.880000,93.000000,93.000000
max,158736.000000,4.663250e+06,2.032640e+07,6.690000,6.680000,2.840000,112.000000,112.000000
